# Ilera-ATLAS · 07_demo: live chat demo with a free Gradio share link

Free alternative to a Hugging Face Space (Gradio Spaces on free CPU now need a PRO subscription).
Opens the Ilera-ATLAS chat page (`serve/space/ui.py`: colour-coded answers, examples in English / Hausa /
Yoruba, streaming text, red-flag safety layer) behind a public `https://xxxx.gradio.live` link.

| `BACKEND` | Hardware | Speed | Uses GPU quota |
|---|---|---|---|
| `"gpu"` | Kaggle GPU T4 / Colab T4 | ~7 s per answer | yes |
| `"cpu"` | Kaggle **CPU** (4 cores, 30 GB) | ~1–2 min per answer (text streams as it is written) | **no** |
| `"auto"` | GPU if one is attached, else CPU | | |

The CPU backend runs the quantised `ilera-atlas-GGUF` (Q4_K_M) with llama.cpp's prebuilt `llama-server`
(no compiling).

* The link is printed below **and saved to the Hugging Face dataset** as `demo/share_url.txt`.
* It stays up for `HOURS` (Kaggle sessions end after 12 h; share links after 72 h). Re-run for a new link.
* Questions/answers are logged **anonymously** to `ilera-data/logs/` (`LOG = True`). Testers must not type
  names or personal details.
* Optional password: `DEMO_PASSWORD` (user name `ilera`).

**Kaggle:** Accelerator *GPU T4 x2* (gpu) or *None* (cpu) · Internet On · secret `HF_TOKEN` ·
**Save Version → Save & Run All (Commit)** so it keeps running with your laptop closed.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"           # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
BACKEND = "auto"                        # "gpu", "cpu" or "auto"
MODEL = f"{HF_USER}/ilera-atlas-lora-runB"        # GPU backend: fine-tuned LoRA adapter
GGUF_REPO = f"{HF_USER}/ilera-atlas-GGUF"         # CPU backend: quantised model
HOURS = 8                               # how long to keep the demo up
LOG = True                              # anonymous interaction log -> ilera-data/logs/
DEMO_PASSWORD = ""                      # e.g. "naic2026" -> login ilera / naic2026; "" = open link
MAX_NEW = 450                           # GPU; the CPU backend uses CPU_MAX_NEW (slower)
CPU_MAX_NEW = 320
GITHUB_RAW = "https://raw.githubusercontent.com/OkeyAmy/Ilera/ilera-atlas-pipeline/serve/space"

In [ ]:
import os, subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception:
    HAS_GPU = False
USE_GPU = BACKEND == "gpu" or (BACKEND == "auto" and HAS_GPU)
assert not (BACKEND == "gpu" and not HAS_GPU), "BACKEND = 'gpu' but no GPU is attached"
pkgs = ["gradio>=5", "openai"] + (["unsloth"] if USE_GPU else [])
if USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("backend:", "GPU" if USE_GPU else "CPU (GGUF + llama-server)", "| packages installed")

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    import glob                                     # API runs: token in a private Kaggle dataset
    for p in glob.glob("/kaggle/input/**/hf_token.txt", recursive=True):
        return open(p).read().strip()
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
assert HF_TOKEN, "No HF_TOKEN found"
os.environ["HF_TOKEN"] = HF_TOKEN

## 1. Chat page, answer format, red flags and safety layer (same files as the Space)

In [ ]:
import csv, io, re, urllib.request, importlib.util
from huggingface_hub import hf_hub_download

for f in ("ui.py", "redflags.csv"):                 # single source of truth: the GitHub repo
    urllib.request.urlretrieve(f"{GITHUB_RAW}/{f}", f)
spec = importlib.util.spec_from_file_location("ilera_ui", "ui.py")
ui = importlib.util.module_from_spec(spec); spec.loader.exec_module(ui)

SYSTEM = open(hf_hub_download(DATA_REPO, "config/system_prompt.txt", repo_type="dataset", token=HF_TOKEN),
              encoding="utf-8").read().strip()
FLAGS = sorted({v.strip().lower() for row in csv.DictReader(open("redflags.csv", encoding="utf-8"))
                for v in row.values() if v and v.strip()}, key=len, reverse=True)

def red_flag(question):
    q = " " + re.sub(r"\s+", " ", question.lower()) + " "
    return next((f for f in FLAGS if f in q), None)

def force_urgent(answer):
    return ui.ACTION_RE.sub("ACTION: URGENT", answer, count=1) if ui.ACTION_RE.search(answer) else "ACTION: URGENT\n" + answer

assert red_flag("baby has convulsions") == "convulsion" and force_urgent("ACTION: TREAT\nx").startswith("ACTION: URGENT")
print(len(FLAGS), "red-flag phrases; UI module loaded; safety self-check passed")

## 2. Load the model (GPU: N-ATLaS + LoRA · CPU: GGUF in llama-server)

In [ ]:
import time, threading, json, glob, tarfile, shutil

if USE_GPU:
    import gc, torch
    from unsloth import FastLanguageModel
    from transformers import TextIteratorStreamer
    globals().pop("model", None); gc.collect(); torch.cuda.empty_cache()
    model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=2048, load_in_4bit=True, token=HF_TOKEN)
    FastLanguageModel.for_inference(model)
    model.generation_config.max_length = None

    def stream_answer(question):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
        streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
        kw = dict(**enc, max_new_tokens=MAX_NEW, do_sample=False, repetition_penalty=1.1, streamer=streamer,
                  pad_token_id=tok.pad_token_id or tok.eos_token_id)
        threading.Thread(target=model.generate, kwargs=kw, daemon=True).start()
        out = ""
        for piece in streamer:
            out += piece
            yield out
else:
    # prebuilt llama.cpp CPU server from the official GitHub releases (no compiling)
    rel = json.load(urllib.request.urlopen(urllib.request.Request(
        "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10", headers={"User-Agent": "ilera"})))
    asset = next(a for r in rel for a in r["assets"] if re.fullmatch(r"llama-b\d+-bin-ubuntu-x64\.tar\.gz", a["name"]))
    print("llama.cpp:", asset["name"])
    urllib.request.urlretrieve(asset["browser_download_url"], "llama.tar.gz")
    shutil.rmtree("llama", ignore_errors=True)
    with tarfile.open("llama.tar.gz") as t:
        t.extractall("llama")
    server = next(p for p in glob.glob("llama/**/llama-server", recursive=True))
    os.chmod(server, 0o755)
    from huggingface_hub import list_repo_files
    gguf_name = next(f for f in list_repo_files(GGUF_REPO, token=HF_TOKEN) if f.endswith(".gguf"))
    gguf = hf_hub_download(GGUF_REPO, gguf_name, token=HF_TOKEN)
    env = dict(os.environ, LD_LIBRARY_PATH=os.path.dirname(os.path.abspath(server)) + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
    proc = subprocess.Popen([os.path.abspath(server), "-m", gguf, "--host", "127.0.0.1", "--port", "8088",
                             "-c", "2048", "-t", str(os.cpu_count() or 4), "--parallel", "1"],
                            env=env, stdout=open("llama-server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(180):                            # wait until the model is loaded
        try:
            if json.load(urllib.request.urlopen("http://127.0.0.1:8088/health", timeout=5)).get("status") == "ok":
                break
        except Exception:
            pass
        assert proc.poll() is None, open("llama-server.log").read()[-2000:]
        time.sleep(5)
    from openai import OpenAI
    client = OpenAI(base_url="http://127.0.0.1:8088/v1", api_key="none")

    def stream_answer(question):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
        out = ""
        for ch in client.chat.completions.create(model="ilera", messages=msgs, temperature=0, max_tokens=CPU_MAX_NEW,
                                                 stream=True, extra_body={"repeat_penalty": 1.1}):
            out += ch.choices[0].delta.content or ""
            yield out

t0 = time.time()
for answer in stream_answer("A 2-year-old has fever and fast breathing. What should I do?"):
    pass
print(f"test answer in {time.time() - t0:.0f}s:\n{answer}")

## 3. Anonymous interaction log (user-test evidence)

In [ ]:
import datetime, uuid
from pathlib import Path
scheduler = None
LOG_DIR = Path("demo_logs"); LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"interactions-{datetime.date.today()}-{uuid.uuid4().hex[:6]}.jsonl"
if LOG:
    from huggingface_hub import CommitScheduler
    scheduler = CommitScheduler(repo_id=DATA_REPO, repo_type="dataset", folder_path=LOG_DIR, path_in_repo="logs",
                                every=5, token=HF_TOKEN)

def log(question, answer, flag):
    if scheduler is None:
        return
    with scheduler.lock, open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"time": datetime.datetime.utcnow().isoformat(timespec="seconds"), "question": question,
                            "answer": answer, "red_flag": flag, "backend": "gpu" if USE_GPU else "cpu"},
                           ensure_ascii=False) + "\n")
print("logging to", f"{DATA_REPO}/logs/{LOG_FILE.name}" if LOG else "(off)")

## 4. Start the chat page and publish the share link

In [ ]:
from huggingface_hub import HfApi

def respond(question):
    # yields (text, final, flag) for ui.build_demo: streamed text first, then the finished answer
    flag = red_flag(question)
    text = ""
    for text in stream_answer(question):
        yield text, False, None
    final = force_urgent(text) if flag else text
    log(question, final, flag)
    yield final, True, flag

demo, launch_kw = ui.build_demo(respond)
demo.queue(default_concurrency_limit=1)
_, local_url, share_url = demo.launch(share=True, prevent_thread_lock=True, **launch_kw,
                                      auth=("ilera", DEMO_PASSWORD) if DEMO_PASSWORD else None)
print("\n" + "=" * 70 + f"\n  ILERA-ATLAS DEMO: {share_url}\n" + "=" * 70)
until = datetime.datetime.utcnow() + datetime.timedelta(hours=HOURS)
note = (f"{share_url}\nbackend {'GPU' if USE_GPU else 'CPU'}; started {datetime.datetime.utcnow():%Y-%m-%d %H:%M} UTC, "
        f"up until about {until:%Y-%m-%d %H:%M} UTC\n")
open("share_url.txt", "w").write(note)
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="share_url.txt", path_in_repo="demo/share_url.txt",
                                  repo_id=DATA_REPO, repo_type="dataset", commit_message="demo share link")
print("saved to", f"{DATA_REPO}/demo/share_url.txt")

## 5. Keep the demo running

In [ ]:
end = time.time() + HOURS * 3600
while time.time() < end:
    time.sleep(600)
    print(time.strftime("%H:%M"), "demo up:", share_url, flush=True)
if scheduler is not None:
    scheduler.trigger().result()        # final log upload
demo.close()
print("demo stopped")